# 04 — Carbon and alkalinity additions in the complete model

**Learning goal:** use matched controls to explain how carbon and alkalinity
inputs propagate through gas exchange, carbonate chemistry, circulation and
sediment response.

**Core time: 40 minutes.** Predict (5), map the two forcings (10), run and inspect
budgets (10), read selected panels and explain the response (15). The model verified in 03 is supplied.
The two short coding blocks specify forcing amounts and connection endpoints.
Plots, integration and inventory audits are supplied.

Attribution and feedback experiments are in a separate
[optional notebook](extensions/04_attribution_and_feedbacks.ipynb).
[Teaching goals](../../TEACHING_GOALS.md).

This notebook includes the aggregate OA/OAE analysis formerly labelled Part II
in the extension: full supplied response figures, a guided reading route, and
matched anomalies. The two coding tasks and four short answers remain the required
work; no panel-by-panel report or additional model runs are required.

In [7]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from esbmtk import Signal, Source, Species2Species
from model import initialize_model, postprocess_carbonate_horizons, run_model
from presets import load_boudreau_parameters, make_pump_variant
from reservoir_inputs import reservoir_inventory_rows
from model_inputs import read_model_tables

DATA = ROOT / 'data' / 'Boudreau_2010'
WORKBOOK = DATA / 'model_definition.xlsx'
P = load_boudreau_parameters(WORKBOOK)
input_tables = read_model_tables(WORKBOOK)
STATE = DATA / 'steady_state'
PULSE_FILE = DATA / 'IS92a-scenario.csv'
DIGITIZED = DATA / 'digitized'
REFERENCE_SCALE = 0.877
PULSE_START = 1800.0
REFERENCE_CARBON_PMOL = 335.3560189847107
OAE_TARGET_PMOL = 10.0
OAE_SCALE = REFERENCE_SCALE * OAE_TARGET_PMOL / REFERENCE_CARBON_PMOL
from teaching_audits import audit_complete_model

## 1. Reuse the verified model and its stationary state

The workbook owns geometry, box-specific T/S/P, transport, gas exchange and
process rates. The displayed input tables are the same definitions inspected
in 03. Keep these benchmark values for the core experiment. Every case receives
an independent copy. The archived restart replaces workbook initial concentrations;
changing geometry, chemistry or baseline rates requires a new stationary restart.

The constant-pump model retains weathering, dissolution and burial. We compare
each forced run with a control having identical parameters and initial state.
Only the external forcing changes. Carbon and TA budgets include boundary fluxes.

The OA benchmark uses the archived IS92a evolution followed by a Gaussian-like
decline, with scale 0.877 and no terrestrial uptake. Each case runs for 3800 model
years with a one-month maximum step. OAE reuses the input's shape and timing at
a different amplitude. The benchmark comparison checks model reproduction;
it is not an independent observational validation of its mechanisms.

In [8]:
display(pd.DataFrame(input_tables['OceanReservoirs']).set_index('Box ID'))
display(pd.DataFrame(input_tables['Atmosphere']).set_index('Box ID'))
display(pd.DataFrame(input_tables['BoundaryNodes']).set_index('Box ID'))
display(pd.DataFrame(input_tables['TransportConnections']).sort_values('Order'))
display(pd.DataFrame(input_tables['GasExchangeConnections']).sort_values('Order'))
display(pd.DataFrame(input_tables['ProcessParameters']).set_index('Parameter'))
print('Derived PIC:', P['pic_export'], '; weathering TA:', P['weathering_ta'])

,Description,Area (m2),Volume (m3),Temperature (degC),Salinity,Pressure (bar),Initial DIC (umol/kg),Initial TA (umol/kg)
Box ID,,,,,,,,
H_b,High-latitude surface,50000000000000,17600000000000000,2.0,35,17.6,2153,2345
L_b,Low-latitude surface,285000000000000,28500000000000000,21.5,35,5.0,1952,2288
D_b,Deep ocean,336000000000000,1290000000000000000,2.0,35,240.0,2291,2399


,Total air (mol),Initial CO2 (ppm),Role
Box ID,,,
CO2_At,177860000000000000000,280,Prognostic CO2


,Type,Species,Role
Box ID,,,
Fw,Source,"DIC, TA",External weathering supply
Fb,Sink,"DIC, TA",Permanent burial boundary


,Order,source,sink,flux_id,Parameter,Species
0,1,H_b,D_b,mix_down,mixing,"DIC, TA"
1,2,D_b,H_b,mix_up,mixing,"DIC, TA"
2,3,L_b,H_b,thc,thc,"DIC, TA"
3,4,H_b,D_b,thc,thc,"DIC, TA"
4,5,D_b,L_b,thc,thc,"DIC, TA"


,Order,Atmosphere,Surface,Species,Parameter
0,1,CO2_At,H_b,CO2,piston_velocity
1,2,CO2_At,L_b,CO2,piston_velocity


,Value,Unit,Role,Description
Parameter,,,,
thc,25.00000,Sverdrup,Prescribed transport,Water transport on every leg of the overturnin...
mixing,30.00000,Sverdrup,Prescribed transport,Equal exchange in both directions between high...
poc_export,200.00000,Tmol/yr,Prescribed export,POC carbon export from low-latitude surface to...
rain_ratio,0.30000,1,Prescribed ratio,PIC/POC carbon-export ratio at unit pump stren...
weathering_dic,12.00000,Tmol/yr,Prescribed boundary flux,Carbonate weathering: one DIC and two TA equiv...
alpha,0.60000,1,Benchmark coefficient,Carbonate compensation coefficient
z0,-200.00000,m,Benchmark elevation,Carbonate compensation reference elevation
piston_velocity,4.80000,m/d,Prescribed exchange rate,Common gas transfer velocity at both surface b...
opt_k_carbonic,13.00000,1,Chemistry choice,PyCO2SYS carbonic-acid constant option


Derived PIC: 60 Tmol/yr ; weathering TA: 24 Tmol/yr


## 2. Predict the response chain before running

Trace the diagram in this order:

**External input → surface carbonate chemistry and gas exchange → circulation
to depth → carbonate dissolution/burial and sediment memory.**

OA adds atmospheric carbon; idealized OAE adds surface TA and directly adds no
DIC. Predict the sign of atmospheric CO2 and surface pH changes, then what a
change in deep carbonate ion might do to dissolution and burial. Revisit this
prediction in the four final answers; no separate written submission is needed.

For the supplied sediment panels, distinguish three diagnostics:

| Symbol | Meaning | How to read the response |
| --- | --- | --- |
| $z_{sat}$ | Calcite saturation horizon, where saturation is 1 | Follows deep-water chemistry |
| $z_{cc}$ | Compensation depth, where modern carbonate rain is completely dissolved | Depends on chemistry and carbonate rain |
| $z_{snow}$ | Boundary of the existing reactive carbonate sediment | Retains memory of past conditions and can lag |

Model depths are positive downward; panel e plots their negative values as
elevation, so a curve moving upward means a shallower horizon. Detailed sediment
equations remain optional. POC and PIC export rates are fixed in these cases;
the partition between dissolution and net burial can still change.

<!-- BEGIN SOLUTION -->
OA raises atmospheric CO2; uptake increases surface DIC and lowers pH and carbonate
ion. Deep-water acidification tends to shoal the saturation/compensation horizons,
increase dissolution and reduce net burial. OAE initially raises TA, pH and
carbonate ion, encouraging atmospheric carbon uptake; its later deep response
tends to deepen the chemical horizons, suppress dissolution and enhance net burial.
The snowline can lag because existing sediments must dissolve or accumulate.
Removing 1 DIC and 2 TA through carbonate burial can erode part of the OAE response.
These are pathway predictions: use the coupled trajectories to assess their timing.
<!-- END SOLUTION -->

## 3. Exercise 04.1: specify the two forcing inventories

Retain the benchmark OA forcing: the archived IS92a-shaped input adds about
4025 Gt C after model year 1800 (12 g C/mol). OAE uses the same shape and timing
but adds 10 Pmol TA equivalents over that interval. These are an OA benchmark
and an idealized OAE experiment, not equal-amplitude interventions.

Assign `oa_target_mol` and `oae_target_mol` in mol C and mol TA equivalents.
One Gt is 10^15 g; one Pmol is 10^15 mol. The supplied scaling code retains
the archived pulse, including its small pre-1800 tail. Inventory reports show
both the post-1800 benchmark amount and the full input used in the model budget.

In [9]:
# Exercise 04.1: convert the prescribed amounts to the model's mol units.
# BEGIN SOLUTION
oa_target_mol = 4025.0 * 1e15 / 12.0
oae_target_mol = 10.0 * 1e15
# END SOLUTION
# Supplied conversion from inventory to the archived signal's scale factor.
# Keep its exact reference normalization; 4025 Gt is a rounded comparison target.
np.testing.assert_allclose(oa_target_mol / 1e15, REFERENCE_CARBON_PMOL, rtol=1e-3)
oae_scale = REFERENCE_SCALE * oae_target_mol / (REFERENCE_CARBON_PMOL * 1e15)

## 4. Exercise 04.2: map external arrows to code

Inside the two branches below, assign `forcing_species` and `forcing_target`:
use the atmospheric CO2 species/reservoir for OA, and TA in the low-latitude
surface for OAE. This is a boundary input from a `Source`; the control has none.
The Signal construction and its connection are supplied after your choices.
No pH is prescribed; it is a calculated response.

In [10]:
def build_complete_case(forcing=None):
    # Supplied model and stationary restart; pumps remain fixed.
    params = make_pump_variant(
        base=P, solubility_strength=1.0, soft_tissue_strength=1.0,
        carbonate_strength=1.0, soft_tissue_feedback=False, carbonate_feedback=False,
    )
    model = initialize_model(params, stop='3800 yr', max_timestep='1 month')
    model.read_state(directory=str(STATE))
    if forcing is None:
        return model
    # Exercise 04.2: set the species and receiving state in both cases.
    if forcing == 'OA':
        # BEGIN SOLUTION
        forcing_species, forcing_target = model.CO2, model.CO2_At
        # END SOLUTION
    elif forcing == 'OAE':
        # BEGIN SOLUTION
        forcing_species, forcing_target = model.TA, model.L_b.TA
        # END SOLUTION
    else:
        raise ValueError(forcing)
    scale = {'OA': REFERENCE_SCALE, 'OAE': oae_scale}[forcing]
    signal = Signal(name='external_input', species=forcing_species, register=model,
                    filename=str(PULSE_FILE), scale=scale)
    source = Source(name='external_source', species=forcing_species)
    connection = Species2Species(source=source, sink=forcing_target,
                                rate='0 mol/yr', signal=signal, id='external_input')
    model.teaching_signal = signal
    model.teaching_connection = connection
    # Preserve the exact solver input for the supplied continuous budget audit.
    model.teaching_signal_time = model.time.copy()
    model.teaching_signal_flux = signal.m.copy()
    if forcing == 'OA':
        model.carbon_signal = signal
    else:
        model.alkalinity_signal = signal
    return model

fixed_cases = {label: build_complete_case(None if label == 'control' else label)
               for label in ('control', 'OA', 'OAE')}
assert fixed_cases['OA'].teaching_connection.sink is fixed_cases['OA'].CO2_At
assert fixed_cases['OAE'].teaching_connection.sink is fixed_cases['OAE'].L_b.TA
display(pd.DataFrame(reservoir_inventory_rows(fixed_cases['control'])).set_index('Box'))


ESBMTK 0.14.3.1.post0  
 Copyright (C) 2020 - 2026  Ulrich G.Wortmann
This program comes with ABSOLUTELY NO WARRANTY
This is free software, and you are welcome to redistribute it
under certain conditions; See the LICENSE file for details.

If you use ESBMTK for your research, please cite:

Wortmann et al. 2025, https://doi.org/10.5194/gmd-18-1155-2025


ESBMTK 0.14.3.1.post0  
 Copyright (C) 2020 - 2026  Ulrich G.Wortmann
This program comes with ABSOLUTELY NO WARRANTY
This is free software, and you are welcome to redistribute it
under certain conditions; See the LICENSE file for details.

If you use ESBMTK for your research, please cite:

Wortmann et al. 2025, https://doi.org/10.5194/gmd-18-1155-2025


ESBMTK 0.14.3.1.post0  
 Copyright (C) 2020 - 2026  Ulrich G.Wortmann
This program comes with ABSOLUTELY NO WARRANTY
This is free software, and you are welcome to redistribute it
under certain conditions; See the LICENSE file for details.

If you use ESBMTK for your research, please cit

,Density (kg/m3),Water mass (kg),Initial carbon (mol),Initial TA (mol eq)
Box,,,,
H_b,1028.798771,1.810686e+19,3.896917e+16,4.252578e+16
L_b,1024.575722,2.920041e+19,5.667813e+16,6.664200e+16
D_b,1038.983451,1.340289e+21,3.076045e+18,3.221934e+18


## 5. Supplied forcing and conservation checks

Before running, inspect the integrated amount and species. After running, the
audit adds atmosphere plus ocean carbon using ESBMTK water masses. It integrates
external input + weathering - net burial; the TA audit uses the linked 1:2
carbonate stoichiometry. The supplied audit evaluates the same carbonate flux
law as the solver; integrating its saved values introduces a small numerical
tolerance. Internal transport and gas exchange cancel in the sum.
Budget errors should be small before you interpret any response.

In [11]:
from teaching_audits import integrate_forcing_history

forcing_rows = {}
for label, target in (('OA', oa_target_mol), ('OAE', oae_target_mol)):
    case = fixed_cases[label]
    time, flux = case.teaching_signal_time, case.teaching_signal_flux
    total = integrate_forcing_history(time, flux, time[-1])
    after_start = total - integrate_forcing_history(time, flux, PULSE_START)
    np.testing.assert_allclose(after_start, target, rtol=1e-3)
    forcing_rows[label] = {'post-1800 input (Pmol C or TA eq)': after_start / 1e15,
                           'whole-run input (Pmol C or TA eq)': total / 1e15}
display(pd.DataFrame(forcing_rows).T)

for label, case in fixed_cases.items():
    print('Running', label)
    run_model(case)
    postprocess_carbonate_horizons(case)
display(pd.DataFrame({label: audit_complete_model(case, label)
                      for label, case in fixed_cases.items()}).T)

,post-1800 input (Pmol C or TA eq),whole-run input (Pmol C or TA eq)
OA,335.357579,340.172309
OAE,10.000047,10.143617


Running control
status=0
message=The solver successfully reached the end of the integration interval.


 Execution took 11.75 CPU seconds, wall time = 11.91 seconds

Running OA
status=0
message=The solver successfully reached the end of the integration interval.


 Execution took 12.33 CPU seconds, wall time = 12.35 seconds

Running OAE
status=0
message=The solver successfully reached the end of the integration interval.


 Execution took 11.97 CPU seconds, wall time = 12.01 seconds



,max carbon error / initial stock,max TA error / initial stock,external input (Pmol C or TA eq)
control,1.084733e-08,2.096911e-08,0.000000
OA,8.335061e-06,1.108447e-06,340.172309
OAE,7.444546e-08,2.116669e-07,10.143617


## 6. Follow the complete OA/OAE response in supplied figures

The following figures restore the eight-panel view of the complete model.
They use the already-integrated cases; run the supplied plotting calls unchanged.
Solid lines are this model. OA's dotted lines are the archived published comparison;
OAE has no reference overlay. These are absolute trajectories; the next figure
subtracts the matching control to isolate the forcing response.

| Panel | Diagnostic | Connection to the model diagram |
| --- | --- | --- |
| a | DIC in all three ocean boxes | Carbon enters, exchanges and circulates |
| b | TA in all three ocean boxes | TA input, transport and carbonate transfers |
| c | pH in all three boxes | Chemistry calculated from DIC and TA |
| d | Low-/high-latitude gas exchange | Carbon transfer between atmosphere and surfaces |
| e | Saturation/compensation horizons and snowline | Chemical response and sediment memory |
| f | Atmospheric CO2 | Finite atmospheric carbon reservoir |
| g | External carbon or TA input | Prescribed forcing, with different units/amplitudes |
| h | Dissolution and net burial | Return to water versus loss from the active system |

**Guided reading (within the 15-minute interpretation allocation):**

1. Follow **g → f/c**: identify the imposed input and the atmospheric/surface response.
2. Compare surface and deep curves in **a/c**, then inspect **e/h** for the slower
   response and any snowline lag. Do not assume the final state is equilibrated.
3. Use the matched-anomaly figure below to compare OA and OAE. Consult **b/d**
   when explaining TA redistribution or gas exchange; no separate answers for
   every panel are required. Read axis scales: the forcings are not equal in amount.

In [12]:
from teaching_plots import plot_figure4
plot_figure4(fixed_cases['OA'], 'OA', reference=True,
             digitized=DIGITIZED, pulse_start=PULSE_START)
plt.show()

ImportError: cannot import name 'plot_figure4' from 'teaching_plots' (D:\ALK\TA\BGC\ESBMTK-practicals\teaching_plots.py)

In [ ]:
plot_figure4(fixed_cases['OAE'], 'OAE', pulse_start=PULSE_START)
plt.show()

## 7. Compare forced-minus-control responses

For every quantity use $\Delta Y(t)=Y_{forced}(t)-Y_{control}(t)$.
Read four diagnostic groups: atmospheric CO2, surface pH, deep DIC, and the
dissolution/burial response. Compare signs and the timing of the largest changes;
the deep ocean and sediments need not have equilibrated by the end of the run.

Both forced cases use the same fixed-pump parameters, boundary processes and
initial state as the unforced control. Absolute benchmark agreement and matched
anomalies answer different questions: reproduction versus response to the input.

In [ ]:
from teaching_plots import plot_matched_responses
plot_matched_responses(fixed_cases, pulse_start=PULSE_START)

## 8. Explain the coupled response (four short answers)

1. Use panels g, f, c and a to trace the OA input from the atmosphere through
   surface chemistry to depth. Identify its entry connection in the code and
   distinguish the earlier surface response from the later deep response.
2. Use OAE and its matched anomalies to explain why adding TA without DIC changes
   atmospheric CO2. Why is OAE not OA with every sign reversed?
3. Use panels e/h to connect deep chemistry, dissolution/net burial and the slower
   sediment-memory response. How do the 1:2 DIC–TA transfers modify the perturbation?
4. Which comparison isolates forcing, and which checks benchmark reproduction?
   Name one prescribed input and two calculated outputs.

<!-- BEGIN SOLUTION -->
OA enters CO2_At through teaching_connection; gas exchange moves carbon into surface
DIC, changing calculated pH, and circulation transports the perturbation to depth.
The atmosphere and surface respond before the deep ocean. OAE changes carbonate
partitioning by adding TA, lowering aqueous CO2 and encouraging uptake; it has a
different species and amount from OA, so the responses are not mirror images.
Acidification tends to increase dissolution and reduce net burial; enhanced TA
tends to oppose those changes. Dissolution returns 1 DIC and 2 TA and net burial
removes them, modifying the initial perturbation. The snowline can lag the chemical
horizons because sediment inventory carries memory. Export rates themselves stay
fixed here. Forced-minus-control anomalies isolate the imposed input within the
model; the dotted reference curves check benchmark reproduction. Input species,
amount and time history are prescribed; atmospheric CO2, pH and sediment responses
are conditional outputs. The plotted endpoint need not be a new equilibrium.
<!-- END SOLUTION -->

**Finish the core here.** Submit the two forcing choices, checked budget and four
short explanations. The full response figures are supplied evidence, not extra
coding or a separate report. The [optional extension](extensions/04_attribution_and_feedbacks.ipynb)
contains Part I's tagged attribution and Part III's feedback hypotheses.